# 4 · Agente

**Módulo 1 · Introducción.** El proyecto del módulo es un agente de cálculo simbólico. El router de la lección 3 deja decidir al modelo una sola vez y termina con el resultado de la herramienta, que el modelo nunca lee. Esta lección cambia una arista, $\texttt{tools} \to \texttt{END}$ por $\texttt{tools} \to \texttt{modelo}$: el modelo lee cada resultado y vuelve a decidir, hasta que responde sin pedidos. Ese grafo es el agente.

La arista nueva cierra un ciclo, y con el ciclo vuelve la pregunta de la lección 1: ¿la ejecución termina? Se define el agente, se cuentan sus supersteps, se prueba que ninguna variante garantiza que termine y se le agrega una guardia con `RemainingSteps`, que lo hace llegar a `END` por sí mismo, sin `GraphRecursionError`.

**Dónde corre.** Las secciones 1 a 5 y la profundización no usan el modelo y corren en el PC, con modelos guionados. La sección 6 usa `qwen3:8b`, que corre en Kaggle con una GPU; el [índice del módulo](README.md) explica cómo abrir el notebook allí.

Requisito: `pip install "langgraph>=1.2" "langchain-ollama>=1.1" "sympy>=1.14"`. Las salidas de este notebook son de langgraph 1.2.17, langgraph-prebuilt 1.1.1, langchain-core 1.7.0, langchain-ollama 1.1.0, sympy 1.14.0 y Ollama 0.40.13 con `qwen3:8b`, en una GPU T4 de Kaggle.

**Preparación.** En Kaggle, la celda siguiente instala las dependencias y Ollama, arranca el servidor y descarga `qwen3:8b`, en poco más de un minuto. En el PC no hace nada.

In [ ]:
import os
import subprocess
import time
import urllib.request

EN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ


def correr(comando: str) -> None:
    """Corre un comando de shell y, si falla, muestra el final de su error."""
    r = subprocess.run(comando, shell=True, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"falló: {comando}\n{r.stderr[-2000:]}")


def preparar_kaggle() -> None:
    """Instala las dependencias y Ollama, arranca el servidor y descarga qwen3:8b."""
    correr("pip install -q 'langgraph==1.2.17' 'langgraph-prebuilt==1.1.1' 'langchain-core==1.7.0'"
           " 'langchain-ollama==1.1.0' 'sympy==1.14.0'")
    correr("(command -v zstd || (apt-get -qq update && apt-get -qq install -y zstd))"
           " && curl -fsSL https://ollama.com/install.sh | sh")
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        try:
            urllib.request.urlopen("http://127.0.0.1:11434/api/version")
            break
        except OSError:
            time.sleep(1)
    else:
        raise RuntimeError("el servidor de Ollama no respondió en 60 s")
    correr("ollama pull qwen3:8b")


if EN_KAGGLE:
    preparar_kaggle()

In [ ]:
from importlib.metadata import version

for paquete in ("langgraph", "langgraph-prebuilt", "langchain-core", "langchain-ollama", "sympy"):
    print(paquete, version(paquete))

## 1. El problema

Por la Proposición 3.5, en el router el nodo del modelo corre una sola vez, antes de que exista cualquier respuesta de herramienta. Un pedido como «Deriva x³ y después integra el resultado» necesita dos decisiones, y la segunda depende del resultado de la primera: para pedir una primitiva de $3x^2$, el modelo tiene que haber leído $3x^2$. En la lección 3, `qwen3:8b` escribió los dos pedidos a la vez, los dos con $x^3$, y el router terminó en $x^4/4$.

Falta volver al modelo después de `tools`. Basta cambiar la arista fija $\texttt{tools} \to \texttt{END}$ del router por $\texttt{tools} \to \texttt{modelo}$. El grafo queda con un ciclo, $\texttt{modelo} \to \texttt{tools} \to \texttt{modelo}$, del que la ejecución sale por la arista condicional cuando el modelo responde sin pedidos.

**Mapa de la lección.** El agente con «Deriva x³ y después integra el resultado», un superstep a la vez. El grafo se define en la sección 2; la 3 cuenta sus supersteps, la 4 muestra que puede no terminar y la 5 le agrega la guardia que lo hace terminar. La sección 6 lo corre con el modelo.

<img src="../../imagenes/diagramas/modulo-1/agente.svg" alt="El agente: START lleva a modelo; de modelo, una arista condicional va a tools si el último mensaje trae pedidos y a END si no; tools vuelve a modelo. Con «Deriva x³ y después integra el resultado», el modelo pide derivar(x**3), tools escribe 3*x**2, el modelo lo lee y pide integrar(3*x**2), tools escribe x**3 y el modelo responde con texto, así que la ejecución va a END en el superstep 5" width="560">

Las herramientas, `linea` y el nodo `tools_seguro` son los de la lección 3:

In [ ]:
from sympy import Symbol, diff, integrate, solve
from sympy.parsing.sympy_parser import convert_xor, parse_expr, standard_transformations

TRANSFORMACIONES = standard_transformations + (convert_xor,)


def leer(texto: str):
    """Convierte texto en sintaxis de SymPy en una expresión; «^» vale como potencia."""
    return parse_expr(texto, transformations=TRANSFORMACIONES)


def derivar(expresion: str, variable: str = "x") -> str:
    """Deriva una expresión respecto de una variable.

    Args:
        expresion: expresión en sintaxis de SymPy, por ejemplo x**2*sin(x).
        variable: variable respecto de la que se deriva.
    """
    return str(diff(leer(expresion), Symbol(variable)))


def integrar(expresion: str, variable: str = "x") -> str:
    """Calcula una primitiva de una expresión, sin constante de integración.

    Args:
        expresion: expresión en sintaxis de SymPy, por ejemplo 1/(1 + x**2).
        variable: variable de integración.
    """
    return str(integrate(leer(expresion), Symbol(variable)))


def resolver(ecuacion: str, variable: str = "x") -> str:
    """Resuelve una ecuación en una incógnita y devuelve la lista de soluciones.

    Args:
        ecuacion: ecuación en sintaxis de SymPy, por ejemplo x**2 - 2 = 0. Sin signo =, se iguala a cero.
        variable: incógnita.
    """
    izquierda, _, derecha = ecuacion.replace("==", "=").partition("=")  # «==», como en Python, vale como «=»
    return str(solve(leer(izquierda) - leer(derecha or "0"), Symbol(variable)))


HERRAMIENTAS = [derivar, integrar, resolver]

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.messages.tool import tool_call
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

tools_seguro = ToolNode(HERRAMIENTAS, handle_tool_errors=True)


def linea(m) -> str:
    """Rol y contenido de un mensaje; sus pedidos si es de rol ai, y su γ y estado si es de rol tool."""
    if m.type == "ai" and m.tool_calls:
        return "ai    pide " + ", ".join(f"{p['name']}({p['args']}) γ={p['id'][:8]}" for p in m.tool_calls)
    if m.type == "tool":
        return f"tool  {m.content!r}  γ={m.tool_call_id[:8]}  {m.status}"
    return f"{m.type:5} {m.content!r}"